In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("/Users/yuxin/Documents/NYU DS/NYU-DS/2026 Fall/ML in Finance/Project/codebase/finance_term_project/data/processed/labeled_data.csv", low_memory=False)
data.drop(columns=['Unnamed: 0'], inplace = True)
data.head(5)

,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,...,profit,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations,label,pred_start_dt
0,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,1908.0,...,-81288.0,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0,0,2012-06-01
1,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,860.0,...,-107382.0,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0,0,2009-06-01
2,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,860.0,...,-77819.0,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0,0,2010-06-01
3,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,620.0,...,36733.0,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0,0,2008-06-01
4,520288,2010-12-31,28.0,SRL,14.0,NaN,2010,130973.0,1136731.0,946.0,...,-83593.0,NaN,3973.0,-4.08,-22.52,398756.0,-897381.0,-2104.0,0,2011-06-01


In [ ]:
data.columns

Index(['id', 'stmt_date', 'HQ_city', 'legal_struct', 'ateco_sector',
       'def_date', 'fs_year', 'asst_intang_fixed', 'asst_tang_fixed',
       'asst_fixed_fin', 'asst_current', 'AR', 'cash_and_equiv', 'asst_tot',
       'eqty_tot', 'eqty_corp_family_tot', 'liab_lt', 'liab_lt_emp',
       'debt_bank_st', 'debt_bank_lt', 'debt_fin_st', 'debt_fin_lt', 'AP_st',
       'AP_lt', 'debt_st', 'debt_lt', 'rev_operating', 'COGS',
       'prof_operations', 'goodwill', 'inc_financing', 'exp_financing',
       'prof_financing', 'inc_extraord', 'taxes', 'profit', 'days_rec',
       'ebitda', 'roa', 'roe', 'wc_net', 'margin_fin', 'cf_operations',
       'label', 'pred_start_dt'],
      dtype='str')

In [2]:
#adding sector names into the dataset
industries = pd.read_csv("/Users/yuxin/Documents/NYU DS/NYU-DS/2026 Fall/ML in Finance/Project/codebase/finance_term_project/data/interim/ateco_sector.csv", low_memory=False)
industries['code'] = pd.to_numeric(industries['code'])

### Grouping features

In [92]:
from sklearn.model_selection import StratifiedKFold

def fit_oof_target_encoder(df, column="ateco_sector", target="label",
                           n_splits=5, smoothing=20, random_state=42):
    """
    Returns:
      encoded_train: leakage-safe, out-of-fold target encoding for df[column]
      full_mapping: mapping fit on all training rows; use this for validation/test data
      global_mean: fallback value for unseen or missing categories
    """
    global_mean = df[target].mean()
    encoded_train = pd.Series(index=df.index, dtype=float)

    folds = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    for train_idx, valid_idx in folds.split(df, df[target]):
        fold_train = df.iloc[train_idx]
        fold_mean = fold_train[target].mean()

        stats = fold_train.groupby(column)[target].agg(["sum", "count"])

        # Smoothed category default rate:
        # small categories are pulled toward the overall default rate.
        mapping = (stats["sum"] + smoothing * fold_mean) / (
            stats["count"] + smoothing
        )

        encoded_train.iloc[valid_idx] = (
            df.iloc[valid_idx][column]
            .map(mapping)
            .fillna(fold_mean)  # unseen/missing category
        )

    # Fit one final mapping on all training data for validation/test transformation.
    full_stats = df.groupby(column)[target].agg(["sum", "count"])
    full_mapping = (full_stats["sum"] + smoothing * global_mean) / (
        full_stats["count"] + smoothing
    )

    return encoded_train, full_mapping, global_mean


def transform_target_encoder(df, column, mapping, global_mean):
    """Apply a mapping learned from training data only."""
    return df[column].map(mapping).fillna(global_mean)

#### businee and structural risk

In [93]:
#industry code
mg_data = data.copy()

mg_data["ateco_sector_te"], ateco_mapping, overall_default_rate = (
    fit_oof_target_encoder(
        mg_data,
        column="ateco_sector",
        target="label",
        n_splits=5,
        smoothing=20,
    )
)
mg_data = mg_data.drop(columns=["ateco_sector"])
mg_data.head()

,id,stmt_date,HQ_city,legal_struct,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,asst_current,...,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations,label,pred_start_dt,ateco_sector_te
0,520288,2011-12-31,28.0,SRL,NaN,2011,67537.0,1137566.0,1908.0,733496.0,...,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0,0,2012-06-01,0.032706
1,520288,2008-12-31,28.0,SRL,NaN,2008,256438.0,1181416.0,860.0,622992.0,...,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0,0,2009-06-01,0.032820
2,520288,2009-12-31,28.0,SRL,NaN,2009,194046.0,1152014.0,860.0,615252.0,...,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0,0,2010-06-01,0.033359
3,520288,2007-12-31,28.0,SRL,NaN,2007,126603.0,1127807.0,620.0,484753.0,...,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0,0,2008-06-01,0.032706
4,520288,2010-12-31,28.0,SRL,NaN,2010,130973.0,1136731.0,946.0,609723.0,...,NaN,3973.0,-4.08,-22.52,398756.0,-897381.0,-2104.0,0,2011-06-01,0.033359


In [94]:
#legal construct - one hot encoding
original_col = "legal_struct"
if original_col in mg_data.columns:
    mg_data = pd.get_dummies(
        mg_data,
        columns=["legal_struct"],
        prefix="legal",
        dtype=int,
    )
mg_data.head()

#If we are putting this dataset into the model, we need to drop one column to prevent multi collinearity!

,id,stmt_date,HQ_city,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,asst_current,AR,...,cf_operations,label,pred_start_dt,ateco_sector_te,legal_SAA,legal_SAU,legal_SPA,legal_SRL,legal_SRS,legal_SRU
0,520288,2011-12-31,28.0,NaN,2011,67537.0,1137566.0,1908.0,733496.0,283013.0,...,-849.0,0,2012-06-01,0.032706,0,0,0,1,0,0
1,520288,2008-12-31,28.0,NaN,2008,256438.0,1181416.0,860.0,622992.0,387653.0,...,-3881.0,0,2009-06-01,0.032820,0,0,0,1,0,0
2,520288,2009-12-31,28.0,NaN,2009,194046.0,1152014.0,860.0,615252.0,388260.0,...,32618.0,0,2010-06-01,0.033359,0,0,0,1,0,0
3,520288,2007-12-31,28.0,NaN,2007,126603.0,1127807.0,620.0,484753.0,455224.0,...,80039.0,0,2008-06-01,0.032706,0,0,0,1,0,0
4,520288,2010-12-31,28.0,NaN,2010,130973.0,1136731.0,946.0,609723.0,280395.0,...,-2104.0,0,2011-06-01,0.033359,0,0,0,1,0,0


In [95]:
#size
mg_data['size'] = np.log(mg_data['asst_tot'])

In [9]:
#past number of defaults -- ALL ZEROS!
mg_data["def_date"] = pd.to_datetime(mg_data["def_date"])
mg_data["pred_start_dt"] = pd.to_datetime(mg_data["pred_start_dt"])

# A default date is repeated across a firm's financial-statement rows,
# so retain each company-default event only once.
default_events = (
    mg_data.loc[mg_data["def_date"].notna(), ["id", "def_date"]]
    .drop_duplicates()
    .sort_values(["def_date", "id"])
    .copy()
)

# Number the firm's distinct default events in chronological order.
default_events["past_num_defaults"] = (
    default_events.groupby("id").cumcount() + 1
)

# the start of that row's prediction period.
rows = mg_data[["id", "pred_start_dt"]].copy()
rows["_row_order"] = np.arange(len(rows))

history = pd.merge_asof(
    rows.sort_values(["pred_start_dt", "id"]),
    default_events,
    by="id",
    left_on="pred_start_dt",
    right_on="def_date",
    direction="backward",
    allow_exact_matches=False,
)

# Firms with no earlier default receive 0.
mg_data["past_num_defaults"] = (
    history.set_index("_row_order")["past_num_defaults"]
    .reindex(range(len(mg_data)))
    .fillna(0)
    .astype(int)
    .to_numpy()
)

#mg_data.head()
mg_data[mg_data['past_num_defaults']!=0]

,id,stmt_date,HQ_city,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,asst_current,AR,...,pred_start_dt,ateco_sector_te,legal_SAA,legal_SAU,legal_SPA,legal_SRL,legal_SRS,legal_SRU,size,past_num_defaults


#### Solvency and Capital Structure

In [96]:
#equity ratio
mg_data['eqty_ratio'] = mg_data['eqty_tot'] / mg_data['asst_tot']

#debt ratio
mg_data['eqty_ratio'] = (mg_data['debt_st'] + mg_data['debt_lt']) / mg_data['asst_tot']

#negative equity flag
mg_data['neg_eqty_flag'] = np.where(
    mg_data['eqty_tot'] < 0, 1, 0
)

#short-term debt share 
mg_data['st_debt_share'] = mg_data['debt_st'] / (mg_data['debt_st'] + mg_data['debt_lt'])

#long-term debt to capital
mg_data['lt_debt_to_capital'] = mg_data['debt_lt'] / (mg_data['debt_lt'] + mg_data['eqty_tot'])

#bank debt concentration
mg_data['bank_debt_conc'] = (mg_data['debt_bank_st'] + mg_data['debt_bank_lt']) / (mg_data['debt_st'] + mg_data['debt_lt'])


#### Liquidity and Working capital

In [97]:
#cash ratio 
mg_data['cash_ratio'] = mg_data['cash_and_equiv'] / mg_data['asst_tot']

#cash-to-financing-expenses
mg_data['cash_to_financing_exp_ratio'] = mg_data['cash_and_equiv'] / mg_data['exp_financing']

#current asset ratio
mg_data['cur_asst_ratio'] = mg_data['cash_and_equiv'] / mg_data['exp_financing']

#receivables ratio
mg_data['AR_ratio'] = mg_data['AR'] / mg_data['rev_operating']

#change in AR turnover
ordered = mg_data.sort_values(["id", "fs_year"]).copy()
previous_year = ordered.groupby("id")["fs_year"].shift(1)
ordered["change_in_ar"] = (
    ordered.groupby("id")["AR"].diff()
).where(
    ordered["fs_year"].eq(previous_year + 1)
)
mg_data["change_in_ar"] = ordered["change_in_ar"].reindex(mg_data.index)

#paybles ratio
mg_data['payables_ratio'] = mg_data['AP_st'] / mg_data['asst_tot']

#working capitle ratio
mg_data['wc_ratio'] = mg_data['wc_net'] / mg_data['asst_tot']

#current over debt ratio
mg_data['cur_debt_ratio'] = mg_data['asst_current'] / mg_data['debt_st']

#### Profitability and Operating Performance

In [98]:
#we already have roa and roe

#change in ROA
ordered = mg_data.sort_values(["id", "fs_year"]).copy()

previous_year = ordered.groupby("id")["fs_year"].shift()
year_gap = ordered["fs_year"] - previous_year

ordered["annualized_change_in_roa"] = (
    ordered.groupby("id")["roa"].diff() / year_gap
).where(year_gap > 0)

mg_data["annualized_change_in_roa"] = ( #because we have year gaps in the data
    ordered["annualized_change_in_roa"].reindex(mg_data.index)
)

#EBITDA ratio
mg_data['ebitda_ratio'] = mg_data['ebitda'] / mg_data['asst_tot']

#operating margin
mg_data['op_margin'] = mg_data['prof_operations'] / mg_data['rev_operating']

#sales growth
ordered = mg_data.sort_values(["id", "fs_year"]).copy()

previous_revenue = ordered.groupby("id")["rev_operating"].shift()
previous_year = ordered.groupby("id")["fs_year"].shift()

years_between = ordered["fs_year"] - previous_year
revenue_ratio = ordered["rev_operating"] / previous_revenue

ordered["annualized_sales_growth"] = (
    revenue_ratio.pow(1 / years_between) - 1
).where(
    (years_between > 0)
    & (ordered["rev_operating"] > 0)
    & (previous_revenue > 0)
)

mg_data["annualized_sales_growth"] = (
    ordered["annualized_sales_growth"].reindex(mg_data.index)
)

#net profit margin
mg_data['net_prof_margin'] = mg_data['profit'] / mg_data['rev_operating']

#loss flag
mg_data['loss_flag'] = np.where(
    mg_data['profit'] < 0, 1, 0
)


#### Debt-serving and Cash-flow Capacity

In [99]:
#interest coverage
mg_data['int_coverage'] = mg_data['ebitda'] / mg_data['exp_financing']

#cash flow to debb
mg_data['cash_to_debt'] = mg_data['cash_and_equiv'] / (mg_data['debt_st'] + mg_data['debt_lt'])

#debt / ebitda
mg_data['debt_to_ebitda'] = (mg_data['debt_st'] + mg_data['debt_lt']) / mg_data['ebitda']

#financial expenses and financial income are already in the dataset

In [14]:
# Audit engineered features separately from the raw-input missingness table above.
# First-year and invalid-denominator ratio values can legitimately be missing; report them rather than imputing them here.
ENGINEERED_COLS = [
    "ateco_sector_te", "size", "past_num_defaults",
    "eqty_ratio", "neg_eqty_flag", "st_debt_share", "lt_debt_to_capital", "bank_debt_conc",
    "cash_ratio", "cash_to_financing_exp_ratio", "cur_asst_ratio", "AR_ratio", "change_in_ar",
    "payables_ratio", "wc_ratio", "cur_debt_ratio",
    "annualized_change_in_roa", "ebitda_ratio", "op_margin", "annualized_sales_growth",
    "net_prof_margin", "loss_flag", "int_coverage", "cash_to_debt", "debt_to_ebitda",
]
legal_cols = [c for c in mg_data.columns if c.startswith("legal_")]
ENGINEERED_COLS = [*ENGINEERED_COLS, *legal_cols]
assert set(ENGINEERED_COLS).issubset(mg_data.columns)

rows = []
for c in ENGINEERED_COLS:
    values = mg_data[c]
    rows.append({
        "feature": c,
        "n_missing": values.isna().sum(),
        "pct_missing": round(values.isna().mean() * 100, 2),
        "n_infinite": np.isinf(values).sum(),
    })
miss = pd.DataFrame(rows).set_index("feature").sort_index()
miss

,n_missing,pct_missing,n_infinite
feature,,,
AR_ratio,675,0.08,25121
annualized_change_in_roa,224554,26.89,0
annualized_sales_growth,251156,30.08,0
ateco_sector_te,0,0.00,0
bank_debt_conc,116958,14.01,0
cash_ratio,17,0.00,0
cash_to_debt,190,0.02,213
cash_to_financing_exp_ratio,1221,0.15,27944
change_in_ar,242489,29.04,0


In [15]:
# Missing-value handling; see docs/project-update/features/02-missing-value.md
DROP_COLS = ["Unnamed: 0", "eqty_corp_family_tot", "days_rec", "roe"]
NON_NUMERIC = ["stmt_date", "legal_struct", "def_date", "pred_start_dt"]
DEBT_DETAIL = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt"]
# Method B: columns filled with (sector median of column / asst_tot) * asst_tot
B_COLS = [
    "asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "asst_current", "AR", "cash_and_equiv", "goodwill",
    "eqty_tot", "liab_lt", "liab_lt_emp", *DEBT_DETAIL, "debt_st", "debt_lt",
    "rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing",
    "inc_extraord", "taxes", "profit", "ebitda", "cf_operations",
]


def sector_medians(train):
    """Median table for method B, computed from the training data: indexed by ateco_sector, last row "ALL" is the overall median.
    The asst_tot column holds the median amount (safeguard for holdout rows missing asst_tot); the other columns hold median ratios to asst_tot."""
    train = _to_numeric(train)
    ratio = train[B_COLS].div(train["asst_tot"], axis=0)
    ratio["asst_tot"] = train["asst_tot"]
    med = ratio.groupby(train["ateco_sector"]).median()
    med.loc["ALL"] = ratio.median()
    return med


def _to_numeric(df):
    # Blanks, "NA" and other missing-value spellings all become NaN
    df = df.copy()
    for c in df.columns.difference(NON_NUMERIC):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fill_missing(df, medians):
    """Take raw data, return filled data. medians must be the result of sector_medians(training data)."""
    df = _to_numeric(df).drop(columns=DROP_COLS, errors="ignore")

    # Method A: a missing HQ_city becomes its own category
    df["HQ_city"] = df["HQ_city"].map("{:.0f}".format, na_action="ignore").fillna("Unknown")

    # Method C: record the flag before filling
    df["debt_detail_missing"] = df[DEBT_DETAIL].isna().any(axis=1).astype(int)

    # Method D: prof_operations = rev_operating - COGS; restore whichever of the three is missing
    df["rev_operating"] = df["rev_operating"].fillna(df["prof_operations"] + df["COGS"])
    df["COGS"] = df["COGS"].fillna(df["rev_operating"] - df["prof_operations"])
    df["prof_operations"] = df["prof_operations"].fillna(df["rev_operating"] - df["COGS"])

    # Method B: look up the sector medians; sectors absent from training use the overall median
    m = medians.drop(index="ALL").reindex(df["ateco_sector"]).set_axis(df.index).fillna(medians.loc["ALL"])
    df["asst_tot"] = df["asst_tot"].fillna(m["asst_tot"])
    df[B_COLS] = df[B_COLS].fillna(m[B_COLS].mul(df["asst_tot"], axis=0))

    # Method E: restore by formula (after B and D, so every input is already filled)
    fixed = df["asst_intang_fixed"] + df["asst_tang_fixed"] + df["asst_fixed_fin"]
    df["margin_fin"] = df["margin_fin"].fillna(df["eqty_tot"] - fixed)
    df["wc_net"] = df["wc_net"].fillna(df["asst_current"] - df["debt_st"])
    df["roa"] = df["roa"].fillna(df["prof_operations"] / df["asst_tot"] * 100)

    return df


In [16]:
medians = sector_medians(data)
filled = fill_missing(data, medians)

# Raw-input checks: def_date is the only field allowed to remain missing.
assert filled.drop(columns="def_date").isna().sum().sum() == 0
assert filled["pred_start_dt"].notna().all()
num = data.columns.difference(DROP_COLS + NON_NUMERIC + ["HQ_city"])
orig = data[num].notna()
assert (filled[num][orig] == data[num][orig]).sum().sum() == orig.sum().sum()
assert filled["debt_detail_missing"].sum() == data[DEBT_DETAIL].isna().any(axis=1).sum()
assert (filled["HQ_city"] == "Unknown").sum() == data["HQ_city"].isna().sum()

# Engineered-feature checks: their missingness is intentionally reported in `miss` above.
assert set(ENGINEERED_COLS).issubset(mg_data.columns)
assert mg_data["past_num_defaults"].ge(0).all()

# Check: missing-value spellings a holdout may use, and a sector never seen in training
raw = data.head(3).astype(str)
raw.loc[raw.index[0], ["AR", "HQ_city", "asst_tot"]] = ["", " ", "NA"]
raw.loc[raw.index[1], "ateco_sector"] = "999"
out = fill_missing(raw, medians)
assert out.drop(columns="def_date").isna().sum().sum() == 0
assert out["HQ_city"].iloc[0] == "Unknown"
filled.shape


(835016, 43)